In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import json
import os
from PIL import Image
from glob import glob
import numpy as np
from tqdm import tqdm
from torchvision import transforms
from pytorchvideo.models.hub import x3d_xs
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# 디바이스 설정
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)

✅ device: cuda:2


In [ ]:

# 행동 key 리스트
key_list = [
    '물과 비누로 손위생',
    '투약처방과 투약원칙 확인 (손으로 짚어서)',
    '근육주사 약물을 정확한 용량과 방법으로 준비',
    '손소독제로 손위생',
    '대상자의 입원팔찌와 투약카드 대조하여 확인',
    '주사부위 노출 후, 주사부위 선정(삼각근)',
    '물과 비누,알콜젤로 손위생 수행',
    '소독솜으로 닦고, 한손으로 주사바늘 뚜껑 제거',
    '주사바늘 90도로 주사부위 찌름',
    '내관당겨보고, 약물 천천히 주입',
    '삽입각도와 같이 빼고, 주사부위 압박',
    '환의 정리',
    '물과 비누로 손위생 (종료 후)'
]

# 파라미터 정의
params = {
    "image_size": 224,
    "frame_size": 50,
    "num_classes": 2,
    "batch_size": 8,
    "data_path": '../../data/',
    "label_path": "../../data/label/check_list/",
    "image_channel": 3,
    "class_name": key_list[6]  # 테스트할 key 설정
}
params["second"] = f'{params["frame_size"]//5}sec'

In [ ]:
# Transform 정의
trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# 데이터 로딩

file_list = [f"D{str(i+1).zfill(3)}" for i in range(200)]
remove_items = ['D151', 'D159', 'D187', 'D080']
filtered_lst = [item for item in file_list if item not in remove_items]
split = int(len(filtered_lst) * 0.8)
filtered_lst = filtered_lst[split:]
def data_load(filtered_lst):
    train_images = torch.zeros(len(filtered_lst), 3, params['image_channel'], params['frame_size'],
                        params['image_size'], params['image_size'])  # [N, 3, C, T, H, W]
    image_label = []

    for i in range(len(filtered_lst)):
        sample_id = filtered_lst[i]
        with open(params['label_path'] + sample_id + '.json', 'r') as f:
            check = json.load(f)

        if params["class_name"] == '물과 비누,알콜젤로 손위생 수행':
            base_path = params['data_path'] + params["second"] + '/' + params["class_name"] + '/' + sample_id
            image_list_1 = sorted(glob(base_path + '/1/*.png'))
            image_list_2 = [f.replace('/1/', '/2/') for f in image_list_1]
            image_list_3 = [f.replace('/1/', '/3/') for f in image_list_1]
            label = 1 if check['행동']["물과 비누 /알콜젤로 손위생 수행"] else 0
        else:
            base_path = params['data_path'] + params["second"] + '/' + params["class_name"] + '/' + sample_id
            image_list_1 = sorted(glob(base_path + '/1/*.png'))
            image_list_2 = [f.replace('/1/', '/2/') for f in image_list_1]
            image_list_3 = [f.replace('/1/', '/3/') for f in image_list_1]
            label = 1 if check['행동'][params["class_name"]] else 0

        image_label.append(label)

        for j in range(params['frame_size']):
            for vid_idx, image_list in enumerate([image_list_1, image_list_2, image_list_3]):
                img = Image.open(image_list[j]).convert('RGB').resize((params['image_size'], params['image_size']))
                train_images[i, vid_idx, :, j] = trans(img)
    return train_images, image_label

# Custom Dataset 정의
class CustomDataset(Dataset):
    def __init__(self, args, video_tensor, labels):
        self.videos = video_tensor
        self.labels = labels
        self.args = args

    def __getitem__(self, idx):
        video1 = self.videos[idx, 0]
        video2 = self.videos[idx, 1]
        video3 = self.videos[idx, 2]
        label = self.labels[idx]
        return video1, video2, video3, label

    def __len__(self):
        return len(self.videos)

# Train/Test Split


In [ ]:
# 모델 정의

class Multix3d(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        self.backbone1 = x3d_xs(pretrained=pretrained)
        self.backbone2 = x3d_xs(pretrained=pretrained)
        self.backbone3 = x3d_xs(pretrained=pretrained)
        self.classifier = nn.Sequential(
            nn.Linear(400 * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, video1, video2, video3):
        feat1 = self.backbone1(video1)
        feat2 = self.backbone2(video2)
        feat3 = self.backbone3(video3)
        fused = torch.cat([feat1, feat2, feat3], dim=1)
        return self.classifier(fused)
    
def test_model(params, model_path, test_dataloader):
    model = Multix3d(num_classes=params['num_classes'])
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for video1, video2, video3, label in test_dataloader:
            video1 = video1.to(device)
            video2 = video2.to(device)
            video3 = video3.to(device)
            label = label.to(device)

            output = model(video1, video2, video3)
            preds = output.argmax(dim=1)
            true_labels = label.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(true_labels.cpu().numpy())


    # ======================= 🎯 지표 계산 =======================
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    # ======================= 🔥 혼동 행렬 시각화 =======================
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Not performed", "Performed"],
                yticklabels=["Not performed", "Performed"])
    plt.title(f"Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")

    # 저장 경로
    save_dir = f"../../result/action/confusion_matrix"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{params['class_name']}.png")
    plt.tight_layout()
    plt.savefig(save_path)

    # ======================= 📦 Return =======================
    return {
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1_score": round(f1, 4)
    }

In [ ]:
import pandas as pd
df=pd.DataFrame(columns=["Key", "accuracy", "precision", "recall", "f1_score"])
for i in tqdm(range(len(key_list))):
    params["class_name"] = key_list[i]  # 테스트할 key 설정
    train_images, image_label = data_load(filtered_lst)
    test_dataset = CustomDataset(params, train_images, F.one_hot(torch.tensor(image_label), num_classes=params['num_classes']).float())
    test_dataloader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False, drop_last=False)
    model_path = f"../../model/{params['class_name']}/best_model_{params['second']}.pt"
    metrics = test_model(params, model_path, test_dataloader)
    df.loc[i] = [params["class_name"], metrics["accuracy"], metrics["precision"], metrics["recall"], metrics["f1_score"]]
df.to_csv(f"../../result/action/metrics_{params['second']}.csv", index=False, encoding="utf-8-sig")